In [ ]:
# Imports
import os
import shutil
from pathlib import Path
from PIL import Image
import pandas as pd
import random

In [ ]:
# Data mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.listdir('/content/drive/MyDrive')

In [ ]:
# I'll be using the YOLO v8 model from Ultralytics (probably nano, but maybe bigger)
!pip install ultralytics
import ultralytics
ultralytics.checks()


Let's begin phase 1! This is just taking a small bit of data from the SkiTB set and getting it set up to train a YOLO model, so nothing crazy so far.

In [ ]:
# Copy from Drive to local Colab disk (better I/O)
DRIVE_PATH = "/content/drive/MyDrive/SkiTB_data"
LOCAL_PATH = "/content/skitb"

# Resetting local copy
if os.path.exists('/content/skitb'):
    shutil.rmtree('/content/skitb')

print("Copying zip...")
shutil.copy('/content/drive/MyDrive/SkiTB_data/SkiTB_subset.7z', '/content/')
print("Copy done!")

print("Unzipping...")
!7z x /content/SkiTB_subset.7z -o/content/skitb
print("Done!")

print(os.listdir('/content/skitb'))

In [ ]:
# Check dimensions of a sample frame
sample = "/content/skitb/SkiTB/FS/FS0020/frames/00300.jpg"
img = Image.open(sample)
print(img.size)  # (width, height), (1280, 720) in our case


In [ ]:
# Converting images from dataset to YOLO compliant shape
from pathlib import Path

def convert_skitb_to_yolo(sequence_dir, output_dir, img_width=1280, img_height=720, visibility_only=True):
    sequence_dir = Path(sequence_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Work at the SC (single camera) clip level
    sc_dir = sequence_dir / "SC"
    frames_dir = sequence_dir / "frames"

    # Check each single cam
    for clip_dir in sc_dir.iterdir():
        if not clip_dir.is_dir():
            continue

        boxes = (clip_dir / "boxes.txt").read_text().strip().splitlines()
        visibilities = (clip_dir / "visibilities.txt").read_text().strip().splitlines()
        frame_indices = (clip_dir / "frames.txt").read_text().strip().splitlines()

        for i, (box, vis, frame_idx) in enumerate(zip(boxes, visibilities, frame_indices)):

            # Skip occluded frames if visibility_only=True
            # (May change later but most jumps will have most of the
            # rider in frame)
            if visibility_only and vis.strip() == "0":
                continue

            x, y, w, h = map(int, box.strip().split(","))

            # Convert to YOLO normalized center format
            cx = (x + w / 2) / img_width
            cy = (y + h / 2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height

            # Clamp to [0, 1] in case of any annotation edges exceeding frame
            cx = max(0, min(1, cx))
            cy = max(0, min(1, cy))
            w_norm = max(0, min(1, w_norm))
            h_norm = max(0, min(1, h_norm))

            # Label file named to match the frame image
            frame_name = frame_idx.strip().zfill(5)  # e.g. "00027"
            label_path = output_dir / f"{frame_name}.txt"

            # class 0 = person/skier
            with open(label_path, "w") as f:
                f.write(f"0 {cx:.6f} {cy:.6f} {w_norm:.6f} {h_norm:.6f}\n")

# Run it over all sequences (recorded runs)
base_dir = "/content/skitb/SkiTB/FS"
output_base = "/content/skitb/yolo/labels"

for seq in sorted(os.listdir(base_dir)):
    seq_path = os.path.join(base_dir, seq)
    if not os.path.isdir(seq_path):
        continue

    out_path = os.path.join(output_base, seq)
    convert_skitb_to_yolo(seq_path, out_path, img_width=1920, img_height=1080)
    print(f"Converted {seq}")

In [ ]:
# Sanity check that everything looks right

# Check label folders were created
print(os.listdir('/content/skitb/yolo/labels'))

# Peek at one random label file
sample_label = list(Path('/content/skitb/yolo/labels').rglob('*.txt'))[0]
print(f"\nSample file: {sample_label}")
print(sample_label.read_text()[:200])

# Count total label files generated
all_labels = list(Path('/content/skitb/yolo/labels').rglob('*.txt'))
print(f"\nTotal label files: {len(all_labels)}")


We'll also want the images themselves moved over to the yolo folder, so we'll do that now. Rather than copy everything, we'll make symlinks to save space. (There may be a better way to do this where the model can read the images directly from the dataset, but for now this works)

In [ ]:
labels_base = Path('/content/skitb/yolo/labels')
images_base = Path('/content/skitb/yolo/images')
frames_base = Path('/content/skitb/SkiTB/FS')

for seq_label_dir in sorted(labels_base.iterdir()):
    seq_name = seq_label_dir.name
    frames_dir = frames_base / seq_name / "frames"
    images_out = images_base / seq_name
    images_out.mkdir(parents=True, exist_ok=True)

    for label_file in seq_label_dir.glob('*.txt'):
        frame_name = label_file.stem
        src = frames_dir / f"{frame_name}.jpg"
        dst = images_out / f"{frame_name}.jpg"
        if src.exists() and not dst.exists():
            os.symlink(src, dst)

print("Symlinks created!")

In [ ]:
# Trying to get a sense for what the data looks like
# by checking counts of certain fields in the runs we're using
df = pd.read_csv('/content/skitb/SkiTB/FS/FS_data.csv')
# print(df.columns.tolist())
sequences = os.listdir('/content/skitb/SkiTB/FS')
sequences = [s for s in sequences if s.startswith('FS')]

df_subset = df[df['ID'].isin(sequences)]

print(f"Sequences in subset: {len(df_subset)}")

# We can use these counts to figure out how to best stratify the sample
# we're working with
print("\nWeather values:")
print(df_subset['Weather'].value_counts())

print("\nCourseLocation values:")
print(df_subset['CourseLocation'].value_counts())

print("\nAthleteGender values:")
print(df_subset['AthleteGender'].value_counts())

In [ ]:
# Mapping the weather categories to 3 broad groups
# so we can get a good mix of conditions across data splits
weather_map = {
    'Sunny': 'good',
    'Clear': 'good',
    'PartlyCloud': 'ok',
    'MostlyCloudy': 'ok',
    'Overcast': 'bad',
    'Fog': 'bad'
}

df_subset = df_subset.copy()
df_subset['WeatherGroup'] = df_subset['Weather'].map(weather_map)
print(df_subset[['ID', 'Weather', 'WeatherGroup']])

Now we'll actually make the train/val/test split with the sample we're using, and we'll ensure that each split gets a fair mix of each kind of weather condition group (good/ok/bad)

In [ ]:
random.seed(42)  # for reproducibility

# Group sequences by weather group
groups = df_subset.groupby('WeatherGroup')['ID'].apply(list).to_dict()
print("Groups:", {k: len(v) for k, v in groups.items()})

train_seqs, val_seqs, test_seqs = [], [], []

# For each weather group, we assign runs to each split
# in proportion to the size of the group
for group, seqs in groups.items():
    random.shuffle(seqs)
    n = len(seqs)
    n_val = max(1, round(n * 0.15))  # working with 70/15/15 split for now hence the 0.15
    n_test = max(1, round(n * 0.15))
    n_train = n - n_val - n_test

    train_seqs += seqs[:n_train]
    val_seqs += seqs[n_train:n_train + n_val]
    test_seqs += seqs[n_train + n_val:]

# Quick check that everything looks good
for split_name, split_seqs in [('Train', train_seqs), ('Val', val_seqs), ('Test', test_seqs)]:
    weather = df_subset[df_subset['ID'].isin(split_seqs)][['ID', 'WeatherGroup', 'Weather']]
    print(f"\n{split_name}:")
    print(weather.to_string(index=False))

Now that we have a distribution for weather groups, we can move files to the necessary train, val, and test folders that the YOLO model will need

In [ ]:
yolo_base = Path('/content/skitb/yolo')

for split_name, split_seqs in [('train', train_seqs), ('val', val_seqs), ('test', test_seqs)]:
    paths = []
    for seq in split_seqs:
        seq_images = (yolo_base / 'images' / seq).glob('*.jpg')
        paths += [str(p) for p in sorted(seq_images)]

    split_file = yolo_base / f'{split_name}.txt'
    split_file.write_text('\n'.join(paths))
    print(f"{split_name}: {len(paths)} images written to {split_file}")

This is definitely not the best place to do this, but for now we need a yaml file. Given how short it'll be, we'll make it here for now and at some point I'll get it into the folder in a less stupid way

In [ ]:
yaml_content = """path: /content/skitb/yolo
train: train.txt
val: val.txt
test: test.txt

nc: 1
names: ['skier']
"""

with open('/content/skitb/yolo/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print("Written!")

# Verify
with open('/content/skitb/yolo/dataset.yaml', 'r') as f:
    print(f.read())

In [ ]:
# Time for one last verification
yolo_base = Path('/content/skitb/yolo')

# 1. Check yaml exists
print("dataset.yaml exists:", (yolo_base / 'dataset.yaml').exists())

# 2. Check split files exist and have content
for split in ['train', 'val', 'test']:
    split_file = yolo_base / f'{split}.txt'
    lines = split_file.read_text().strip().splitlines()
    print(f"{split}.txt: {len(lines)} images")

# 3. Check images and labels are paired
print("\nChecking image/label pairs...")
mismatches = 0
for split in ['train', 'val', 'test']:
    split_file = yolo_base / f'{split}.txt'
    image_paths = split_file.read_text().strip().splitlines()
    for img_path in image_paths:
        img = Path(img_path)
        label = yolo_base / 'labels' / img.parent.name / (img.stem + '.txt')
        if not label.exists():
            mismatches += 1

print(f"Mismatched pairs: {mismatches}")

# 4. Spot check one label file
sample_img = Path(yolo_base / 'train.txt').read_text().strip().splitlines()[0]
sample_label = yolo_base / 'labels' / Path(sample_img).parent.name / (Path(sample_img).stem + '.txt')
print(f"\nSample label ({sample_label.name}):")
print(sample_label.read_text())

This is the end of the data preprocessing. There wasn't really much to do since we're just using a tiny bit of the SkiTB data to mess around with a model until we can find something promising to use on more data. A lot of this code was written with AI (especially given that it's a lot of "simple" grunt work), but as a future note to myself: the complexity of the work at this point should still be easily understandable, and if that is ever not the case I should review the relevant LLM chats and CSE 446/7 material immediately (will probably add a link or something later).